# Forest Fire Risk Intelligence

This notebook captures the preprocessing, modeling, and evaluation of geospatial forest fire risks.

## Data Cleaning & Preparation

### Code from `src/preprocessing.py`

In [ ]:
import pandas as pd
import numpy as np

def preprocess_data(df):
    """Handle missing values and encode categorical columns."""
    # Fill missing values
    df.fillna(df.median(numeric_only=True), inplace=True)
    df.fillna(df.mode().iloc[0], inplace=True)
    
    # Encode binary columns
    if 'daynight' in df.columns:
        df['daynight'] = df['daynight'].map({'D': 1, 'N': 0})
        
    if 'satellite' in df.columns:
        df['satellite'] = df['satellite'].map({'Terra': 0, 'Aqua': 1})
        
    return df


### Code from `src/feature_engineering.py`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

def get_season(month):
    if month in [12, 1, 2]:
        return 1  # winter
    elif month in [3, 4, 5]:
        return 2  # spring
    elif month in [6, 7, 8]:
        return 3  # summer
    else:
        return 4  # autumn

def create_features(df):
    """Generate temporal, spatial, and risk-based features."""
    # 1. Temporal features
    df['acq_date'] = pd.to_datetime(df['acq_date'])
    df['year'] = df['acq_date'].dt.year
    df['month'] = df['acq_date'].dt.month
    df['day'] = df['acq_date'].dt.day
    
    # 2. Time features
    df['acq_time_str'] = df['acq_time'].astype(int).astype(str).str.zfill(4)
    df['hours'] = df['acq_time_str'].str[:2].astype(int)
    df['minutes'] = df['acq_time_str'].str[2:].astype(int)
    
    # 3. Season
    df['season'] = df['month'].apply(get_season)
    
    # 4. Spatial Clustering
    kmeans = KMeans(n_clusters=15, random_state=42)
    df['region_cluster'] = kmeans.fit_predict(df[['latitude', 'longitude']])
    
    # Map clusters to readable pseudo-states based on centroids
    cluster_names = {
        0: "Punjab & Haryana", 1: "Odisha & Chhattisgarh", 2: "Mizoram & Tripura",
        3: "Western Maharashtra", 4: "Western Madhya Pradesh", 5: "Eastern Madhya Pradesh",
        6: "Tamil Nadu", 7: "UP & Nepal Border", 8: "Meghalaya & Assam",
        9: "Marathwada & Telangana", 10: "Gujarat", 11: "Nagaland & Upper Assam",
        12: "Jharkhand", 13: "Bastar & Eastern Telangana", 14: "Rayalaseema (AP)"
    }
    df['region_name'] = df['region_cluster'].map(cluster_names)
    
    # 5. Fire Activity counts
    df['cluster_fire_count'] = df.groupby('region_cluster')['region_cluster'].transform('count')
    df['monthly_cluster_fire_count'] = df.groupby(['region_cluster', 'month'])['region_cluster'].transform('count')
    
    # 6. Thermal Anomalies
    df['brightness_diff'] = df['brightness'] - df['bright_t31']
    
    # 7. Targets and Categories
    df['high_risk'] = ((df['confidence'] > 80) & (df['frp'] > 20)).astype(int)
    
    df['frp_category'] = pd.cut(
        df['frp'],
        bins=[0, 10, 30, 60, 1000],
        labels=['Low', 'Medium', 'High', 'Extreme']
    )
    
    # Sort and reset index
    df = df.sort_values(['year', 'month', 'day', 'hours', 'minutes']).reset_index(drop=True)
    
    return df


## Visualization & EDA

### Code from `src/visualization.py`

In [ ]:
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap
import os
import seaborn as sns
import pandas as pd

def get_color(risk):
    if risk == 'Low': return 'green'
    if risk == 'Medium': return 'orange'
    return 'red'

def create_fire_map(df, center=[22, 80], zoom=5, sample_size=5000):
    """Create a cleaner Folium HeatMap for fire hotspots."""
    fire_map = folium.Map(location=center, zoom_start=zoom, tiles='CartoDB dark_matter')
    
    sample_df = df.sample(min(len(df), sample_size))
    
    # Heatmap layer
    heat_data = [[row['latitude'], row['longitude'], row['frp']] for index, row in sample_df.iterrows()]
    HeatMap(heat_data, radius=15, blur=10, max_zoom=1).add_to(fire_map)
    
    # Add colored markers for highest risk (optional, just top 100 to avoid clutter)
    extreme_fires = sample_df[sample_df['high_risk'] == 1].nlargest(100, 'frp')
    for _, row in extreme_fires.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4,
            color='red',
            fill=True,
            fill_opacity=0.8,
            popup=f"Risk: High | FRP: {row['frp']}"
        ).add_to(fire_map)
        
    return fire_map

def plot_daily_trend(df):
    """Plot daily fire counts and 7-day moving average."""
    daily_counts = df.groupby('acq_date').size().reset_index(name='fire_count')
    daily_counts['moving_avg_7'] = daily_counts['fire_count'].rolling(7).mean()
    
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(daily_counts['acq_date'], daily_counts['fire_count'], label='Actual Daily Count', alpha=0.4, color='#ff7f0e')
    ax.plot(daily_counts['acq_date'], daily_counts['moving_avg_7'], label='7-Day Moving Average', linewidth=2, color='#d62728')
    ax.set_title("Historical Daily Fire Trend", fontsize=14)
    ax.set_xlabel("Date")
    ax.set_ylabel("Fire Count")
    ax.legend()
    fig.tight_layout()
    return fig

def plot_forecast(forecast):
    """Plot Prophet forecast with confidence intervals."""
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Filter dataframe directly for readability (4 years historical + 6 months forecast)
    max_date = forecast['ds'].max()
    start_date = max_date - pd.DateOffset(years=4, months=6)
    plot_df = forecast[(forecast['ds'] >= start_date) & (forecast['ds'] <= max_date)]
    
    ax.plot(plot_df['ds'], plot_df['yhat'], label='Forecasted Trend', color='#d62728', linewidth=2)
    ax.fill_between(plot_df['ds'], plot_df['yhat_lower'], plot_df['yhat_upper'], color='#ff9896', alpha=0.3, label='Confidence Interval')
    
    # Historical part? Usually prophet's plot method is easier but we do it custom for UI
    historical = plot_df[plot_df['trend'].notnull() & (plot_df['ds'] < plot_df['ds'].max() - pd.Timedelta(days=180))]
    
    ax.set_title("6-Month Wildfire Forecasting Projection", fontsize=14)
    ax.set_xlabel("Date")
    ax.set_ylabel("Projected Fire Count (Monthly)")
    ax.legend()
    fig.tight_layout()
    return fig


### Code from `src/insights.py`

In [ ]:
import pandas as pd

def generate_insights(df, forecast):
    """Generate dynamic AI risk insights based on historical and forecasted data."""
    insights = []
    
    # 1. Historical Peak Season
    monthly_fires = df.groupby(df['acq_date'].dt.month).size()
    peak_month = monthly_fires.idxmax()
    month_names = {1: 'January', 2: 'February', 3: 'March', 4: 'April', 5: 'May', 6: 'June', 7: 'July', 8: 'August', 9: 'September', 10: 'October', 11: 'November', 12: 'December'}
    
    insights.append(f"🔥 Historical data indicates that **{month_names[peak_month]}** is the peak month for wildfire activity.")
    
    # 2. Forecast Trend Analysis
    last_hist_date = df['acq_date'].max()
    future_forecast = forecast[forecast['ds'] > last_hist_date]
    
    if not future_forecast.empty:
        trend_diff = future_forecast['yhat'].iloc[-1] - future_forecast['yhat'].iloc[0]
        if trend_diff > 0:
            insights.append(f"📈 Forecast models predict an **escalating trend** (+{trend_diff:.0f} fires/month) in wildfire activity over the next 6 months.")
        else:
            insights.append(f"📉 Forecast models project a **decrease** in wildfire events for the upcoming 6-month period.")
            
        peak_forecast_month = future_forecast.loc[future_forecast['yhat'].idxmax()]
        insights.append(f"⚠️ Extreme caution advised for **{peak_forecast_month['ds'].strftime('%B %Y')}**, which shows the highest projected risk ({peak_forecast_month['yhat']:.0f} anticipated events).")
    
    # 3. High Risk Regions
    if 'region_name' in df.columns:
        top_region = df[df['high_risk'] == 1]['region_name'].mode()[0]
        insights.append(f"📍 **{top_region}** has historically concentrated the most high-severity thermal anomalies (FRP > 20, Confidence > 80%).")
        
    return insights


## Model Performance & Results

### Code from `src/modeling.py`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def split_data(df, features, target='high_risk'):
    """Split data into training and testing sets."""
    X = df[features]
    y = df[target]
    
    return train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

def train_random_forest(X_train, y_train, n_estimators=100):
    """Train a Random Forest Classifier."""
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    model.fit(X_train, y_train)
    return model

def train_xgboost(X_train, y_train):
    """Train an XGBoost Classifier."""
    model = XGBClassifier(random_state=42, eval_metric='logloss')
    model.fit(X_train, y_train)
    return model


### Code from `src/evaluation.py`

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

def evaluate_model(y_test, y_pred, model_name="Model"):
    """Print classification report and ROC-AUC score."""
    print(f"--- {model_name} Evaluation ---")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    try:
        auc = roc_auc_score(y_test, y_pred)
        print(f"ROC-AUC: {auc:.4f}")
    except ValueError:
        print("ROC-AUC could not be calculated (possibly only one class in y_test).")
    
    return confusion_matrix(y_test, y_pred)


### Code from `src/forecasting.py`

In [ ]:
import pandas as pd
from prophet import Prophet
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def prepare_time_series(df):
    """Aggregate historical MODIS observations monthly."""
    df['acq_date'] = pd.to_datetime(df['acq_date'])
    monthly_data = df.groupby(pd.Grouper(key='acq_date', freq='ME')).agg({
        'frp': 'mean',
        'latitude': 'count', # Count of fires
        'brightness': 'mean',
        'high_risk': 'sum'
    }).reset_index()
    monthly_data.rename(columns={'latitude': 'fire_count'}, inplace=True)
    return monthly_data

def train_prophet_model(df):
    """Train Prophet forecasting model on historical monthly wildfire activity."""
    ts_data = prepare_time_series(df)
    
    # Prophet requires 'ds' (date) and 'y' (target)
    prophet_df = ts_data[['acq_date', 'fire_count']].rename(columns={'acq_date': 'ds', 'fire_count': 'y'})
    
    # Train model
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    model.fit(prophet_df)
    return model, ts_data

def generate_forecast(model, periods=6, freq='ME'):
    """Generate projected wildfire activity trends up to 6 months from today's date."""
    last_date = model.history['ds'].max()
    today = pd.Timestamp.today()
    target_date = today + pd.DateOffset(months=6)
    
    if target_date > last_date:
        diff_months = (target_date.year - last_date.year) * 12 + target_date.month - last_date.month
        periods = max(periods, diff_months)
        
    future = model.make_future_dataframe(periods=periods, freq=freq)
    forecast = model.predict(future)
    return forecast

def evaluate_forecast(model, df):
    """Evaluate forecasting performance using MAE, RMSE."""
    ts_data = prepare_time_series(df)
    prophet_df = ts_data[['acq_date', 'fire_count']].rename(columns={'acq_date': 'ds', 'fire_count': 'y'})
    
    # Use in-sample prediction for metric evaluation
    forecast = model.predict(prophet_df)
    y_true = prophet_df['y']
    y_pred = forecast['yhat']
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

def engineer_forecasting_features(df):
    """Engineer lag-based temporal features and rolling averages."""
    ts_data = prepare_time_series(df)
    ts_data['rolling_avg_3m'] = ts_data['fire_count'].rolling(window=3).mean()
    ts_data['lag_1m'] = ts_data['fire_count'].shift(1)
    ts_data['lag_1y'] = ts_data['fire_count'].shift(12)
    
    # Seasonal indicator
    ts_data['month'] = ts_data['acq_date'].dt.month
    ts_data['is_peak_season'] = ts_data['month'].apply(lambda x: 1 if x in [3, 4, 5] else 0)
    
    return ts_data
